# 04 — Batch Enrich First & Last Authors via SerpAPI

In many scientometric studies the **first author** (most likely the primary researcher) and the **last author** (most likely the senior supervisor) are analysed separately.

This notebook shows how to:
1. Load the example paper dataset
2. Parse first and last author names from the `Authors` column
3. Look up each author on Google Scholar via SerpAPI
4. Fetch their h-index and compute consecutive publishing years
5. Merge the enriched data back and export to CSV

This is a simplified, well-commented version of the main loop in `inspiration/EDA.ipynb`.

In [ ]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from serpapi import GoogleSearch

In [ ]:
load_dotenv('../.env')
SERPAPI_KEY = os.getenv('SERPAPI_KEY')
if not SERPAPI_KEY or SERPAPI_KEY == 'your_serpapi_key_here':
    raise ValueError('SERPAPI_KEY not set.')

REFERENCE_YEAR = 2023

## 1. Load and prepare the dataset

In [ ]:
df = pd.read_csv('../data/example_papers.csv')
df_included = df[df['Inclusion decision'] == 'include'].copy()
print(f'Processing {len(df_included)} included papers')

In [ ]:
def parse_author_name(raw: str) -> str:
    """Convert 'Last, First' to 'First Last'."""
    raw = raw.strip()
    parts = raw.split(',')
    if len(parts) == 2:
        return f'{parts[1].strip()} {parts[0].strip()}'
    return raw


def get_first_last_authors(authors_str: str):
    """Return (first_author, last_author) as display names."""
    if pd.isna(authors_str) or not authors_str.strip():
        return None, None
    parts = [p.strip() for p in authors_str.split(';') if p.strip()]
    first = parse_author_name(parts[0]) if parts else None
    last = parse_author_name(parts[-1]) if len(parts) > 1 else None
    return first, last


df_included[['first_author', 'last_author']] = df_included['Authors'].apply(
    lambda x: pd.Series(get_first_last_authors(x))
)

df_included[['Title', 'first_author', 'last_author']].head()

## 2. Helper functions

In [ ]:
def get_scholar_author_id(name: str, api_key: str) -> str | None:
    """Search Google Scholar for an author by name, return their profile ID."""
    if not name:
        return None
    params = {'engine': 'google_scholar_profiles', 'mauthors': name, 'api_key': api_key}
    profiles = GoogleSearch(params).get_dict().get('profiles', [])
    return profiles[0]['author_id'] if profiles else None


def get_author_metrics(author_id: str, api_key: str) -> dict:
    """Fetch h-index, total citations, and article year list for a Google Scholar author ID."""
    params = {'engine': 'google_scholar_author', 'author_id': author_id, 'sort': 'pubdate', 'num': 100, 'api_key': api_key}
    data = GoogleSearch(params).get_dict()

    h_index = total_citations = None
    for row in data.get('cited_by', {}).get('table', []):
        if 'h_index' in row:
            h_index = row['h_index'].get('all')
        if 'citations' in row:
            total_citations = row['citations'].get('all')

    pub_years = []
    for article in data.get('articles', []):
        year_raw = article.get('year')
        if year_raw:
            try:
                pub_years.append(int(year_raw))
            except ValueError:
                pass

    return {'h_index': h_index, 'total_citations': total_citations, 'pub_years': pub_years}


def consecutive_publishing_years(pub_years: list, end_year: int) -> int:
    years_with_pubs = set(pub_years)
    count = 0
    year = end_year
    while year in years_with_pubs:
        count += 1
        year -= 1
    return count


def classify_seniority(consecutive_years: int) -> str:
    if consecutive_years < 3:
        return 'early'
    elif consecutive_years <= 8:
        return 'mid'
    else:
        return 'senior'

## 3. Main enrichment loop

> **Note:** Each paper requires up to 4 SerpAPI calls (search + profile for first author, same for last author). With 6 papers, this uses ~24 of your monthly quota. Uncomment `time.sleep` calls if you hit rate limits.

In [ ]:
enriched = []

for i, row in df_included.iterrows():
    print(f'\n[{len(enriched)+1}/{len(df_included)}] {str(row["Title"])[:70]}')

    record = {
        'Paper id': row['Paper id'],
        'DOI': row['DOI'],
        'Title': row['Title'],
        'PubYear': row['PubYear'],
    }

    for role, name_col in [('first', 'first_author'), ('last', 'last_author')]:
        name = row[name_col]
        print(f'  {role} author: {name}')

        author_id = get_scholar_author_id(name, SERPAPI_KEY)
        time.sleep(0.5)

        if author_id:
            metrics = get_author_metrics(author_id, SERPAPI_KEY)
            time.sleep(0.5)
            consec = consecutive_publishing_years(metrics['pub_years'], REFERENCE_YEAR)
            seniority = classify_seniority(consec)
            print(f'    → h-index={metrics["h_index"]}, consecutive={consec}, seniority={seniority}')
        else:
            metrics = {'h_index': None, 'total_citations': None}
            consec = None
            seniority = None
            print('    → not found on Google Scholar')

        record[f'{role}_author_name'] = name
        record[f'{role}_author_scholar_id'] = author_id
        record[f'{role}_author_hindex'] = metrics['h_index']
        record[f'{role}_author_total_citations'] = metrics['total_citations']
        record[f'{role}_author_consecutive_years'] = consec
        record[f'{role}_author_seniority'] = seniority

    enriched.append(record)

print('\nDone!')

## 4. Review and export results

In [ ]:
df_enriched = pd.DataFrame(enriched)
df_enriched[['Title', 'first_author_name', 'first_author_hindex', 'first_author_seniority',
             'last_author_name', 'last_author_hindex', 'last_author_seniority']]

In [ ]:
output_path = '../data/example_papers_author_enriched.csv'
df_enriched.to_csv(output_path, index=False)
print(f'Saved to {output_path}')